## Project: Fake/Real News Detection

In [104]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [105]:
current_folder = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

data_path = project_root/"dataset"/"news_data.csv"
df = pd.read_csv(data_path)


/tmp/ipykernel_17987/4270221016.py:9: DtypeWarning: Columns (0: Unnamed: 22, 1: Unnamed: 23, 2: Unnamed: 24, 3: Unnamed: 25, 4: Unnamed: 26, 5: Unnamed: 27, 6: Unnamed: 28, 7: Unnamed: 29, 8: Unnamed: 30, 9: Unnamed: 31, 10: Unnamed: 32, 11: Unnamed: 33, 12: Unnamed: 34, 13: Unnamed: 35, 14: Unnamed: 36, 15: Unnamed: 37, 16: Unnamed: 38, 17: Unnamed: 39, 18: Unnamed: 40, 19: Unnamed: 41, 20: Unnamed: 42, 21: Unnamed: 43, 22: Unnamed: 44, 23: Unnamed: 45, 24: Unnamed: 46, 25: Unnamed: 47, 26: Unnamed: 48, 27: Unnamed: 49, 28: Unnamed: 50, 29: Unnamed: 51, 30: Unnamed: 52, 31: Unnamed: 53, 32: Unnamed: 54, 33: Unnamed: 55, 34: Unnamed: 56, 35: Unnamed: 57, 36: Unnamed: 58, 37: Unnamed: 59, 38: Unnamed: 60, 39: Unnamed: 61, 40: Unnamed: 62, 41: Unnamed: 63, 42: Unnamed: 64, 43: Unnamed: 65, 44: Unnamed: 66, 45: Unnamed: 67, 46: Unnamed: 68, 47: Unnamed: 69, 48: Unnamed: 70, 49: Unnamed: 71, 50: Unnamed: 72, 51: Unnamed: 73, 52: Unnamed: 74, 53: Unnamed: 75, 54: Unnamed: 76, 55: Unnamed: 7

In [106]:
df.head()

,text,label,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136,Unnamed: 137,Unnamed: 138
0,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,U.S. Secretary of State John F. Kerry said Mon...,REAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,It's primary day in New York and front-runners...,REAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [107]:
df.shape

(7795, 139)

In [108]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7795 entries, 0 to 7794
Columns: 139 entries, text to Unnamed: 138
dtypes: str(139)
memory usage: 8.3 MB


In [109]:
df.describe()

,text,label,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 129,Unnamed: 130,Unnamed: 131,Unnamed: 132,Unnamed: 133,Unnamed: 134,Unnamed: 135,Unnamed: 136,Unnamed: 137,Unnamed: 138
count,6929,6755,318,241,179,133,91,65,55,38,...,1,1,1,1,1,1,1,1,1,1
unique,6644,437,315,241,179,133,91,65,55,38,...,1,1,1,1,1,1,1,1,1,1
top,"Killing Obama administration rules, dismantlin...",REAL,etc.,bold way,” Maureen Dowd writes. “Before he jumped into ...,Trump was seen as bombastic,vulgar,a bit of a buffoon and a cave man,but [also] ‘a cheeky brio.’ He was not regard...,not a predator … But he created another chara...,...,rather than acting out of a sober evaluation ...,but to me,more and more,the whole shooting match seems to come down t...,if one could discredit the idea of American e...,then it becomes much easier to have a serious...,a presidential candidate,a senator can stand behind the podium and mak...,it just seems that undertaking a serious crit...,REAL
freq,58,3161,3,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [110]:
df = df[['text','label']]

In [111]:
df.head(3)

,text,label
0,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,U.S. Secretary of State John F. Kerry said Mon...,REAL


In [112]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7795 entries, 0 to 7794
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    6929 non-null   str  
 1   label   6755 non-null   str  
dtypes: str(2)
memory usage: 121.9 KB


#### The dataset was loaded and only the relevent columns were retained for furthur analysis.

In [113]:
df['label'].value_counts()

label
REAL                                                                                                                  3161
FAKE                                                                                                                  3154
 or naturalization or by jus sanguinis – inherited through ancestors/parents                                             3
 for example                                                                                                             2
 etc.                                                                                                                    2
                                                                                                                      ... 
 culminating with the Siege of Jerusalem in 70 CE. In viscous hand to hand combat between the various Jewish sects       1
 after all                                                                                                               1
 Fla. Bide

In [114]:
valid_labels = ['REAL', 'FAKE']

In [115]:
df = df[df['label'].isin(valid_labels)]

In [116]:
df['label'].value_counts()

label
REAL    3161
FAKE    3154
Name: count, dtype: int64

In [117]:
df.head()

,text,label
0,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,It's primary day in New York and front-runners...,REAL


In [118]:
df.isna().sum()

text     0
label    0
dtype: int64

In [119]:
print(df.duplicated().any())
print(df.duplicated().sum())

True
275


In [120]:
df[df.duplicated()]

,text,label
61,"On this day in 1973, J. Fred Buzhardt, a lawye...",REAL
90,"Killing Obama administration rules, dismantlin...",REAL
157,"Killing Obama administration rules, dismantlin...",REAL
233,"On this day in 1973, J. Fred Buzhardt, a lawye...",REAL
410,"Killing Obama administration rules, dismantlin...",REAL
...,...,...
7687,Email \nISIS barbarians used an industrial dou...,FAKE
7693,Email \nNorth Korea’s Foreign Ministry slammed...,FAKE
7710,A verdict in 2017 could have sweeping conseque...,REAL
7730,"Killing Obama administration rules, dismantlin...",REAL


In [121]:
df[df.duplicated()].index

Index([  61,   90,  157,  233,  410,  417,  569,  596,  647,  655,
       ...
       7608, 7628, 7635, 7654, 7660, 7687, 7693, 7710, 7730, 7788],
      dtype='int64', length=275)

In [122]:
df[df.duplicated(keep=False)].drop_duplicates().shape[0]

69

In [123]:
df = df.drop_duplicates(keep='first')

### We found that some of the news is duplicated. Lets find them out.

#### Only valid target labels were retained to ensure correct supervised learning.

## Baseline Sanity Check
#### A baseline approach was use first to assess whether the dataset contains sufficent signal for classification before applying advanced processing techniques.
#### Count Vectorizer will be used for word processing.

In [124]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [125]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words='english')


In [126]:
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [127]:
mnb = MultinomialNB()

In [128]:
y_pred = mnb.fit(X_train_vec, y_train).predict(X_test_vec)

In [129]:
accuracy = accuracy_score(y_test, y_pred)
cls_report = classification_report(y_test, y_pred)
cnf_mat = confusion_matrix(y_test, y_pred)

In [130]:
print(f"Accuracy: {accuracy}")
print(f"Classification Report: {cls_report}")
print(f"confusion Matrix: {cnf_mat}")

Accuracy: 0.8807947019867549
Classification Report:               precision    recall  f1-score   support

        FAKE       0.90      0.86      0.88       612
        REAL       0.86      0.90      0.88       596

    accuracy                           0.88      1208
   macro avg       0.88      0.88      0.88      1208
weighted avg       0.88      0.88      0.88      1208

confusion Matrix: [[528  84]
 [ 60 536]]


## Baseline Sanity Analysis
#### With raw text this kind of result is quite impressive. Accuracy is 90.3%, Recall is 93% , Precision: Real: 89 & Fake: 92. Overall strong model performance. But Count Vectorizer is not suitable for this kind of sensitive model though our model performance is quite good enough. Because Count Vectorizer can't understand the importance of words.

## Natural Language Processing--> Manual

In [131]:
def lower_text(text):
    text = text.lower()
    return text

def remove_punc(text):
    text = re.sub(r"[^\w\s!?]", "", text)
    return text

def remove_whitespace(text):
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [132]:
def normalize_text(text):
    if pd.isna(text):
        return text
    
    text = lower_text(text)
    text = remove_punc(text)
    text = remove_whitespace(text)
    return text

In [133]:
df['text'] = df['text'].apply(normalize_text)

In [134]:
df['text'].head(3)

0    daniel greenfield a shillman journalism fellow...
1    google pinterest digg linkedin reddit stumbleu...
2    us secretary of state john f kerry said monday...
Name: text, dtype: str

In [135]:
X = df['text']
y = df['label']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [136]:
cl_vectorizer = CountVectorizer(stop_words='english')

In [137]:

X_train_vec_c = cl_vectorizer.fit_transform(X_train_c)
X_test_vec_c = cl_vectorizer.transform(X_test_c)

In [138]:
mnb_clean = MultinomialNB()

y_pred_c = mnb_clean.fit(X_train_vec_c, y_train_c).predict(X_test_vec_c)

In [139]:
accuracy_c = accuracy_score(y_test_c, y_pred_c)
cls_report_c = classification_report(y_test_c, y_pred_c)
cnf_mat_c = confusion_matrix(y_test_c, y_pred_c)

In [140]:
print(f"Accuracy(Clean): {accuracy_c}")
print(f"Classification Report: {cls_report_c}")
print(f"Confusion Matrix: {cnf_mat_c}")

Accuracy(Clean): 0.875
Classification Report:               precision    recall  f1-score   support

        FAKE       0.90      0.85      0.87       612
        REAL       0.85      0.90      0.88       596

    accuracy                           0.88      1208
   macro avg       0.88      0.88      0.87      1208
weighted avg       0.88      0.88      0.87      1208

Confusion Matrix: [[520  92]
 [ 59 537]]


## NLP Model Analysis
#### After the NLP pipeline, new model is also close to baseline model. But we overclean the text's, thats why model can't catch full meaning of a word and why it is used. After NLP pipeline implementions, we can assume why Count Vectorizer is not good with clean sensitive texts, what we talked earlier. So lets try TF-IDF and then with Bigram.

In [141]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_df=0.9,
    min_df=5
)

X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

mnb_tf = MultinomialNB()

y_pred_tf = mnb_tf.fit(X_train_tf, y_train).predict(X_test_tf)

In [142]:
accuracy_tf = accuracy_score(y_test, y_pred_tf)
cl_report_tf = classification_report(y_test, y_pred_tf)
cnf_matrix_tf = confusion_matrix(y_test, y_pred_tf)

In [143]:
print(f"Accuracy Score: {accuracy_tf}")
print(f"classification_report: {cl_report_tf}")
print(f"Confusion Matrix: {cnf_matrix_tf}")

Accuracy Score: 0.8733443708609272
classification_report:               precision    recall  f1-score   support

        FAKE       0.91      0.83      0.87       612
        REAL       0.84      0.91      0.88       596

    accuracy                           0.87      1208
   macro avg       0.88      0.87      0.87      1208
weighted avg       0.88      0.87      0.87      1208

Confusion Matrix: [[511 101]
 [ 52 544]]


## TF-IDF Analysis
#### So we've finished TF-IDF, But not that much improvement. Accuracy is like as previous model. Recall has raised, but increase fake news mis(92). Lets try with Bigram.

## N-Grams

In [144]:
#tf-idf vectorizer
tfidf_ngram = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1,2),
    min_df=5,
    max_df=0.9
)

## Implementing Multinomial Naive Bayes Model

In [145]:
X_train_nrg = tfidf_ngram.fit_transform(X_train)
X_test_nrg = tfidf_ngram.transform(X_test)

mnb_nrg = MultinomialNB()
y_pred_nrg = mnb_nrg.fit(X_train_nrg, y_train).predict(X_test_nrg)

In [146]:
acc = accuracy_score(y_test, y_pred_nrg)
cl_rep = classification_report(y_test, y_pred_nrg)
cnf_mat = confusion_matrix(y_test, y_pred_nrg)

In [147]:
print(f"Accuracy: {acc}")
print(f"classification_report: {cl_rep}")
print(f"Confusion Matrix: {cnf_mat}")

Accuracy: 0.8981788079470199
classification_report:               precision    recall  f1-score   support

        FAKE       0.93      0.86      0.90       612
        REAL       0.87      0.94      0.90       596

    accuracy                           0.90      1208
   macro avg       0.90      0.90      0.90      1208
weighted avg       0.90      0.90      0.90      1208

Confusion Matrix: [[526  86]
 [ 37 559]]


## TF-IDF+Bigram Analysis
#### Result is quite good now. Accuracy meets 90.8%. Other metrics is also good. Model miss 86 fake news, but real news detection is improved. 
#### So we can say this model is production friendly.

### Limitations & Future Improvements:
* Implement Logistic Regression / Linear SVM
* Hyperparameter tuning with GridSearchCV
* Lemmatization instead of aggressive cleaning
* Add ROC-AUC for threshold analysis


## Artifacts Loading

In [148]:
import joblib

In [149]:
artifact_dir = project_root/"artifacts"
artifact_dir.mkdir(exist_ok=True)

joblib.dump(
    tfidf_ngram,
    artifact_dir/"tfidf_vectorizer.joblib"
)

joblib.dump(
    mnb_nrg,
    artifact_dir/"news_classifier.joblib"
)

['/home/npro/ml_projects/fake_news_detection/Fake-News-Detection-with-Classical-NLP-Machine-Learning/artifacts/news_classifier.joblib']

In [150]:
list(artifact_dir.iterdir())

[PosixPath('/home/npro/ml_projects/fake_news_detection/Fake-News-Detection-with-Classical-NLP-Machine-Learning/artifacts/tfidf_vectorizer.joblib'),
 PosixPath('/home/npro/ml_projects/fake_news_detection/Fake-News-Detection-with-Classical-NLP-Machine-Learning/artifacts/news_classifier.joblib')]